# Figure 9: Anterograde vs. Retrograde Transport Regimes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import loadmat

plt.rcParams['font.family'] = 'Times New Roman'

# Load Simulation Data

In [ ]:
data_ant = loadmat('./data/ant_sim2.mat')
data_ret = loadmat('./data/ret_sim2.mat')

N_ant = data_ant['model_output'][0,0][0][0,0][0]
N_ret = data_ret['model_output'][0,0][0][0,0][0]

In [ ]:
# Time axis
x1 = np.arange(0, 0.10025, 0.0025)
x2 = np.arange(0.1050, 0.3005, 0.005)
x3 = np.arange(0.31, 1.01, 0.01)
x = np.concatenate((x1, x2, x3)) * 180

max_index = 100  # ~90 days
max_days = 90
day_inc = 30

# Region Labels

In [ ]:
# Abbreviated region labels for hippocampal subnetwork
abbrev = {
    'Dentate gyrus': 'Dent. gyr.',
    'Field CA3': 'CA3',
    'Field CA2': 'CA2',
    'Field CA1': 'CA1',
    'Subiculum, dorsal part': 'Sub. dor.',
    'Subiculum, ventral part': 'Sub. vnt.',
    'Presubiculum': 'Presub.',
    'Parasubiculum': 'Parasub.',
    'Postsubiculum': 'Postsub.',
    'Entorhinal area, lateral part': 'Ent. lat.',
    'Entorhinal area, medial part, dorsal zone': 'Ent. dor.',
    'Retrosplenial area, dorsal part': 'Retro. dor.',
    'Retrosplenial area, ventral part': 'Retro. vnt.',
    'Primary somatosensory area, barrel field': 'Somsens.',
    'Postpiriform transition area': 'Pst. tr.',
}

# Load region labels from atlas
atlas_data_file = './data/CCF_labels.mat'
atlas_data = loadmat(atlas_data_file)

reg1 = np.arange(26, 37, 1)
reg2 = np.arange(78, 81, 1)
reg_hem = np.concatenate((reg1, reg2, np.array([147])))
reg_inds = np.concatenate((reg_hem, reg_hem + 213))

region_labels = []
for i, reg_ind in enumerate(reg_inds):
    name = atlas_data['CCF_labels'][reg_ind][0][0]
    hem = ' RH' if i < len(reg_inds) // 2 else ' LH'
    region_labels.append(name + hem)

def abbreviate(label):
    for suffix in (' RH', ' LH'):
        if label.endswith(suffix):
            label = label[:-len(suffix)]
            break
    return abbrev.get(label, label)

short_labels = [abbreviate(l) for l in region_labels]

# Figure 9a: Tau Concentration Over Time

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

for i in range(30):
    axs[0].plot(x[:max_index], N_ant[i, :max_index])
    axs[1].plot(x[:max_index], N_ret[i, :max_index])

for ax, title in zip(axs, ['Anterograde', 'Retrograde']):
    ax.tick_params(axis='y', labelsize=14)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.yaxis.get_offset_text().set_fontsize(14)
    ax.set_xticks(np.arange(0, max_days+1, day_inc))
    ax.set_xticklabels(np.arange(0, max_days+1, day_inc), fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('Days', fontsize=14)

axs[0].set_ylabel(r'[Tau] ($\mu M$)', fontsize=14)
plt.tight_layout()
plt.savefig('Figure9a.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure 9b: Heatmaps

In [ ]:
# Subsample time indices for heatmap
inds1 = np.arange(0, 41, 4)
inds2 = np.arange(0, 40, 2) + len(x1) + 1
inds3 = np.arange(0, 70, 1) + len(x1) + len(x2)
inds_all = np.concatenate((inds1, inds2, inds3))

fig, axs = plt.subplots(1, 2, figsize=(16, 12))

tick_positions = [0, 24, 50]
tick_labels_hm = ['0', '45', '90']

sns.heatmap(N_ant[:, inds_all][:, :50], ax=axs[0], cbar=False, yticklabels=short_labels)
sns.heatmap(N_ret[:, inds_all][:, :50], ax=axs[1], cbar=True, yticklabels=False)

for ax in axs:
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels_hm, rotation=0)
    ax.tick_params(labelsize=14)
    ax.set_xlabel('Days', fontsize=14)

axs[0].set_title('Anterograde', fontsize=16, fontweight='bold')
axs[1].set_title('Retrograde', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure9b.png', dpi=300, bbox_inches='tight')
plt.show()